# Universidad Peruana Cayetano Heredia (UPCH)
## Pipeline 1: Razonamiento y Búsqueda en Espacio de Estados
### Caso: Minería de Asteroides — Tipo A
**Estudiante:** Leonardo Javier Nunton Fajardo | **Docente:** Yannick Patrick Carrasco Merma

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)

---


## 1. Problema

El caso analizado es la **Minería de Asteroides**, un sector en crecimiento que depende de decisiones
inteligentes sobre qué asteroides vale la pena visitar. El problema central es:

> ¿Cómo puede una IA planificar qué asteroides visitar en qué orden,
> para obtener la mayor cantidad de mineral posible sin quedarse sin combustible?

Para resolverlo vamos a modelarlo como un **problema de búsqueda en espacio de estados**
y aplicar tres algoritmos: **BFS**, **DFS** y **A\***.


## 2. Espacio de Estados

Un "estado" es una fotografía de la misión en un momento dado.
Lo representamos como una tupla con 4 datos:

- **Posición actual:** en qué asteroide (o en la Tierra) estamos
- **Combustible gastado (Δv):** cuánto hemos consumido en km/s
- **Valor acumulado:** cuánto mineral hemos recogido en millones USD
- **Asteroides visitados:** lista de los que ya exploramos

**Estado inicial:** partimos de la Tierra (LEO) con 0 combustible gastado y 0 valor.

**Estado meta:** regresar a la Tierra (LEO) con un valor total ≥ 850 millones USD,
sin haber gastado más de 12.5 km/s de combustible.


In [ ]:
# Importamos las librerías necesarias
# heapq nos da la cola de prioridad para A*
# deque nos da la cola para BFS
from collections import deque
import heapq

print("Librerías cargadas correctamente.")


## 3. Exploración: Los Asteroides Candidatos

Primero definimos los asteroides que el agente puede visitar.
Cada uno tiene:
- **delta_v:** cuánto combustible necesita para llegar desde la Tierra (km/s)
- **valor:** los recursos minerales que contiene (millones USD)


In [ ]:
# Lista de asteroides candidatos (datos simplificados)
asteroides = [
    {"id": "LEO",     "delta_v": 0.0, "valor": 0.0},
    {"id": "Bennu",   "delta_v": 4.2, "valor": 180.0},
    {"id": "Ryugu",   "delta_v": 4.6, "valor": 220.0},
    {"id": "Eros",    "delta_v": 5.4, "valor": 510.0},
    {"id": "Apollo",  "delta_v": 5.8, "valor": 480.0},
    {"id": "Amun",    "delta_v": 6.8, "valor": 920.0},
]

# Mostramos la lista
print(f"{'ID':<10} {'Delta-V (km/s)':<18} {'Valor (M USD)'}")
print("-" * 42)
for a in asteroides:
    print(f"{a['id']:<10} {a['delta_v']:<18} {a['valor']}")


### Análisis:
Los asteroides tipo M como **Amun** tienen el mayor valor (920M USD) pero requieren más combustible (6.8 km/s).
Los tipo C como **Bennu** son más baratos de visitar pero con menor retorno económico.
La IA debe balancear este trade-off.


In [ ]:
# Función que calcula el costo de viajar entre dos cuerpos
# Si estamos en uno y queremos ir a otro, calculamos la diferencia de delta_v
def costo_viaje(origen, destino):
    # Si el origen es LEO, el costo es el delta_v del destino
    if origen == "LEO":
        return destino["delta_v"]
    # Si volvemos a LEO, el costo es el 95% del delta_v del origen (retorno asistido)
    if destino["id"] == "LEO":
        return origen["delta_v"] * 0.95
    # Entre dos asteroides, estimamos por la diferencia de órbita
    return round(abs(origen["delta_v"] - destino["delta_v"]) * 1.5 + 2.5, 2)

# Construimos un diccionario para acceder rápido a cada asteroide por ID
ast_dict = {a["id"]: a for a in asteroides}

# Probamos el cálculo de costo
print("Costo LEO -> Amun:", costo_viaje(ast_dict["LEO"], ast_dict["Amun"]), "km/s")
print("Costo Amun -> LEO:", costo_viaje(ast_dict["Amun"], ast_dict["LEO"]), "km/s")
print("Costo Bennu -> Eros:", costo_viaje(ast_dict["Bennu"], ast_dict["Eros"]), "km/s")


### Análisis:
El agente ya puede calcular cuánto combustible costará cada viaje.
Ahora construiremos los tres algoritmos de búsqueda para encontrar la mejor ruta.


## 4. Búsqueda: Implementación de los Algoritmos

Vamos a implementar **BFS**, **DFS** y **A\*** para que la IA encuentre la ruta óptima.
Cada algoritmo trabaja diferente:

- **BFS:** explora nivel por nivel (primero los vecinos más cercanos)
- **DFS:** profundiza en una rama antes de probar otra
- **A\*:** usa una función de costo `f = g + h` para ir directo a la mejor opción

### Restricciones que todos deben respetar:
- El combustible total no puede superar **12.5 km/s**
- Siempre hay que guardar combustible suficiente para **volver a la Tierra**
- El valor acumulado mínimo para "ganar" es **850 millones USD**


In [ ]:
DV_MAX = 12.5
META_VALOR = 850.0

def obtener_vecinos(posicion, dv_gastado, valor, visitados):
    vecinos = []

    for ast in asteroides:
        if ast["id"] in visitados:
            continue  
        costo = costo_viaje(ast_dict[posicion], ast)
        costo_retorno = ast["delta_v"] * 0.95  

       
        if dv_gastado + costo + costo_retorno <= DV_MAX:
            vecinos.append(ast["id"])

    
    if valor >= META_VALOR and posicion != "LEO":
        costo_retorno = ast_dict[posicion]["delta_v"] * 0.95
        if dv_gastado + costo_retorno <= DV_MAX:
            vecinos.append("LEO")

    return vecinos

print("Desde LEO con 0 km/s gastados, podemos ir a:")
print(obtener_vecinos("LEO", 0.0, 0.0, {"LEO"}))


In [ ]:
# ============================================================
# BFS — Búsqueda en Anchura 
# ============================================================
def busqueda_bfs():
    cola = deque()
    cola.append(("LEO", 0.0, 0.0, ["LEO"], {"LEO"}))

    nodos = 0
    while cola:
        posicion, dv, valor, ruta, visitados = cola.popleft()
        nodos += 1

        if posicion == "LEO" and len(ruta) > 1 and valor >= META_VALOR:
            return {"algoritmo": "BFS", "exito": True, "ruta": ruta,
                    "dv": round(dv, 2), "valor": valor, "nodos": nodos}

        for siguiente in obtener_vecinos(posicion, dv, valor, visitados):
            costo = costo_viaje(ast_dict[posicion], ast_dict[siguiente])
            nuevo_valor = valor + ast_dict[siguiente]["valor"]
            cola.append((siguiente, dv + costo, nuevo_valor,
                         ruta + [siguiente], visitados | {siguiente}))

    return {"algoritmo": "BFS", "exito": False, "nodos": nodos}

resultado_bfs = busqueda_bfs()
print("BFS:", resultado_bfs)


### Análisis BFS:
BFS encuentra la solución explorando por niveles. Es completo (siempre encuentra si existe solución)
pero puede expandir muchos nodos porque no sabe cuál es el camino más barato.


In [ ]:
# ============================================================
# DFS — Búsqueda en Profundidad (va hasta el fondo primero)
# ============================================================
def busqueda_dfs():
    pila = [("LEO", 0.0, 0.0, ["LEO"], {"LEO"})]

    nodos = 0
    while pila:
        posicion, dv, valor, ruta, visitados = pila.pop()  # pop() saca el último
        nodos += 1

        if posicion == "LEO" and len(ruta) > 1 and valor >= META_VALOR:
            return {"algoritmo": "DFS", "exito": True, "ruta": ruta,
                    "dv": round(dv, 2), "valor": valor, "nodos": nodos}

        for siguiente in obtener_vecinos(posicion, dv, valor, visitados):
            costo = costo_viaje(ast_dict[posicion], ast_dict[siguiente])
            nuevo_valor = valor + ast_dict[siguiente]["valor"]
            pila.append((siguiente, dv + costo, nuevo_valor,
                         ruta + [siguiente], visitados | {siguiente}))

    return {"algoritmo": "DFS", "exito": False, "nodos": nodos}

resultado_dfs = busqueda_dfs()
print("DFS:", resultado_dfs)


### Análisis DFS:
DFS profundiza rápidamente en una sola rama. Puede encontrar una solución rápido,
pero no garantiza que sea la ruta con menor consumo de combustible.


In [ ]:
# A* — Búsqueda Informada con Heurística
# f(n) = g(n) + h(n)
# g(n) = combustible gastado hasta ahora
# h(n) = estimación del combustible mínimo para volver a LEO
# ============================================================
def busqueda_a_estrella():
    frontera = []
    estado_inicial = ("LEO", 0.0, 0.0, ["LEO"], {"LEO"})
    heapq.heappush(frontera, (0.0, 0.0, estado_inicial))

    nodos = 0
    while frontera:
        f, g, estado = heapq.heappop(frontera)
        posicion, dv, valor, ruta, visitados = estado
        nodos += 1

        if posicion == "LEO" and len(ruta) > 1 and valor >= META_VALOR:
            return {"algoritmo": "A*", "exito": True, "ruta": ruta,
                    "dv": round(dv, 2), "valor": valor, "nodos": nodos}

        for siguiente in obtener_vecinos(posicion, dv, valor, visitados):
            costo = costo_viaje(ast_dict[posicion], ast_dict[siguiente])
            nuevo_dv = dv + costo
            nuevo_valor = valor + ast_dict[siguiente]["valor"]

            
            h = ast_dict[siguiente]["delta_v"] * 0.95 if siguiente != "LEO" else 0.0
            nuevo_f = nuevo_dv + h

            nuevo_estado = (siguiente, nuevo_dv, nuevo_valor,
                            ruta + [siguiente], visitados | {siguiente})
            heapq.heappush(frontera, (nuevo_f, nuevo_dv, nuevo_estado))

    return {"algoritmo": "A*", "exito": False, "nodos": nodos}

resultado_astar = busqueda_a_estrella()
print("A*:", resultado_astar)


### Análisis A*:
A* usa la heurística `h(n)` para estimar el costo restante mínimo.
Eso le permite ir directamente a los estados más prometedores,
expandiendo mucho menos nodos que BFS y DFS.


## 5. Resultados: Comparación de los Tres Algoritmos


In [ ]:
# Comparamos los tres resultados en una tabla simple
import matplotlib.pyplot as plt

resultados = [resultado_bfs, resultado_dfs, resultado_astar]

print("=" * 55)
print(f"{'Algoritmo':<8} {'Exito':<8} {'Delta-V':<12} {'Valor (M$)':<12} {'Nodos'}")
print("=" * 55)
for r in resultados:
    exito = "Sí" if r["exito"] else "No"
    dv = str(r.get("dv", "-"))
    valor = str(r.get("valor", "-"))
    print(f"{r['algoritmo']:<8} {exito:<8} {dv:<12} {valor:<12} {r['nodos']}")


In [ ]:
# Gráfico 1: Nodos expandidos (eficiencia)
algoritmos = [r["algoritmo"] for r in resultados]
nodos = [r["nodos"] for r in resultados]
dv_total = [r.get("dv", 0) for r in resultados]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Barras de nodos
colores = ["green", "orange", "steelblue"]
axes[0].bar(algoritmos, nodos, color=colores)
axes[0].set_title("Nodos expandidos por algoritmo")
axes[0].set_ylabel("Cantidad de nodos")
axes[0].set_xlabel("Algoritmo")
for i, v in enumerate(nodos):
    axes[0].text(i, v + 1, str(v), ha="center", fontweight="bold")

# Barras de Delta-V consumido
axes[1].bar(algoritmos, dv_total, color=colores)
axes[1].axhline(y=DV_MAX, color="red", linestyle="--", label=f"Límite: {DV_MAX} km/s")
axes[1].set_title("Combustible total consumido (Delta-V)")
axes[1].set_ylabel("km/s")
axes[1].set_xlabel("Algoritmo")
axes[1].legend()
for i, v in enumerate(dv_total):
    axes[1].text(i, v + 0.1, f"{v} km/s", ha="center", fontweight="bold")

plt.tight_layout()
plt.show()


## Conclusiones

1. **BFS** exploró más nodos porque revisa todas las opciones nivel por nivel, sin priorizar.
2. **DFS** puede encontrar una solución rápido pero la ruta puede consumir más combustible.
3. **A\*** fue el más eficiente: al usar la heurística `h(n)`, fue directo hacia las rutas
   con menor costo propulsivo, gastando menos combustible y expandiendo menos nodos.

Para una misión real de minería de asteroides, **A\*** sería el algoritmo más adecuado
porque maximiza los recursos obtenidos con el menor gasto de combustible posible.
